In [97]:
import pandas as pd

## Data Exploration

In [98]:
# Data Information:  Start Date: 2012-01-20     End Date: 2026-11-02     Area of Interest: California

modis = pd.read_csv("data/raw/fire_archive_M-C61_846974.csv")
viirs = pd.read_csv("data/raw/fire_archive_SV-C2_846975.csv")

print("===== MODIS =====")
print("Shape:", modis.shape)
print("Columns:", modis.columns.tolist())
print(modis.head())
print("\n")

print("===== VIIRS =====")
print("Shape:", viirs.shape)
print("Columns:", viirs.columns.tolist())
print(viirs.head())

===== MODIS =====
Shape: (226541, 15)
Columns: ['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_t31', 'frp', 'daynight', 'type']
   latitude  longitude  brightness  scan  track    acq_date  acq_time  \
0   33.4453  -114.6864       305.8   1.1    1.1  2012-01-20      1839   
1   33.4919  -114.6754       308.4   1.1    1.1  2012-01-20      1839   
2   32.9750  -116.5996       351.0   1.0    1.0  2012-01-20      1839   
3   32.9766  -116.6101       379.6   1.0    1.0  2012-01-20      1839   
4   33.5324  -115.9580       307.7   1.0    1.0  2012-01-20      1839   

  satellite instrument  confidence  version  bright_t31    frp daynight  type  
0     Terra      MODIS          36     6.03       292.8    4.8        D     0  
1     Terra      MODIS          59     6.03       289.1    8.2        D     0  
2     Terra      MODIS          96     6.03       288.1   66.5        D     0  
3     Terra      MOD

In [99]:
print("MODIS confidence values:")
print(modis["confidence"].value_counts().sort_index())

print("\nVIIRS confidence values:")
print(viirs["confidence"].value_counts())

print("\nDate coverage:")
print("MODIS:", modis["acq_date"].min(), "to", modis["acq_date"].max())
print("VIIRS:", viirs["acq_date"].min(), "to", viirs["acq_date"].max())

MODIS confidence values:
confidence
0       1417
1         16
2         31
3         74
4         98
       ...  
96      2562
97      2642
98      2635
99      2523
100    61093
Name: count, Length: 101, dtype: int64

VIIRS confidence values:
confidence
n    889659
h     72100
l     44044
Name: count, dtype: int64

Date coverage:
MODIS: 2012-01-20 to 2026-08-31
VIIRS: 2012-01-20 to 2026-07-31


In [100]:
print("MODIS satellites:")
print(modis["satellite"].value_counts())

print("\nMODIS versions:")
print(modis["version"].value_counts())

print("\nVIIRS satellites:")
print(viirs["satellite"].value_counts())

print("\nVIIRS versions:")
print(viirs["version"].value_counts())

MODIS satellites:
satellite
Terra    115169
Aqua     111372
Name: count, dtype: int64

MODIS versions:
version
6.03     197351
61.03     29190
Name: count, dtype: int64

VIIRS satellites:
satellite
SNPP    1005803
Name: count, dtype: int64

VIIRS versions:
version
2    1005803
Name: count, dtype: int64


In [101]:
print(
    modis.groupby(["version", "satellite"])["acq_date"]
    .agg(["min", "max", "count"])
)

                          min         max   count
version satellite                                
6.03    Aqua       2012-01-20  2022-12-22   96567
        Terra      2012-01-20  2022-12-22  100784
61.03   Aqua       2023-01-04  2026-08-31   14805
        Terra      2023-01-03  2026-08-31   14385


### Add A new "Resolution" Column

resolution info retrieved from: https://www.earthdata.nasa.gov/s3fs-public/2023-03/FIRMS_OnePager_2022_Prnt-Web.pdf?VersionId=1.pHw_vC487jLkt_aDy.bMvHQxDFC44f&

In [102]:
# Create working copies
modis_clean = modis.copy()
viirs_clean = viirs.copy()

# Add sensor identification
modis_clean["sensor"] = "MODIS"
viirs_clean["sensor"] = "VIIRS"

# Add approximate nominal spatial resolution
modis_clean["nominal_resolution_m"] = 1000
viirs_clean["nominal_resolution_m"] = 375

# Check the results
print("MODIS:")
print(modis_clean[["satellite", "sensor", "nominal_resolution_m"]].head())

print("\nVIIRS:")
print(viirs_clean[["satellite", "sensor", "nominal_resolution_m"]].head())

MODIS:
  satellite sensor  nominal_resolution_m
0     Terra  MODIS                  1000
1     Terra  MODIS                  1000
2     Terra  MODIS                  1000
3     Terra  MODIS                  1000
4     Terra  MODIS                  1000

VIIRS:
  satellite sensor  nominal_resolution_m
0      SNPP  VIIRS                   375
1      SNPP  VIIRS                   375
2      SNPP  VIIRS                   375
3      SNPP  VIIRS                   375
4      SNPP  VIIRS                   375


In [103]:
## Combine the two datasets
fire_data = pd.concat(
    [modis_clean, viirs_clean],
    ignore_index=True
)

print("Combined shape:", fire_data.shape)

print("\nRecords by sensor:")
print(fire_data["sensor"].value_counts())

print("\nColumns:")
print(fire_data.columns.tolist())

Combined shape: (1232344, 17)

Records by sensor:
sensor
VIIRS    1005803
MODIS     226541
Name: count, dtype: int64

Columns:
['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_t31', 'frp', 'daynight', 'type', 'sensor', 'nominal_resolution_m']


In [104]:
fire_data.head()

,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,type,sensor,nominal_resolution_m
0,33.4453,-114.6864,305.8,1.1,1.1,2012-01-20,1839,Terra,MODIS,36,6.03,292.8,4.8,D,0,MODIS,1000
1,33.4919,-114.6754,308.4,1.1,1.1,2012-01-20,1839,Terra,MODIS,59,6.03,289.1,8.2,D,0,MODIS,1000
2,32.9750,-116.5996,351.0,1.0,1.0,2012-01-20,1839,Terra,MODIS,96,6.03,288.1,66.5,D,0,MODIS,1000
3,32.9766,-116.6101,379.6,1.0,1.0,2012-01-20,1839,Terra,MODIS,100,6.03,291.0,154.9,D,0,MODIS,1000
4,33.5324,-115.9580,307.7,1.0,1.0,2012-01-20,1839,Terra,MODIS,0,6.03,292.1,3.8,D,0,MODIS,1000


In [105]:
# Convert acquisition date
fire_data["acq_date"] = pd.to_datetime(fire_data["acq_date"])

# Convert HHMM acquisition time into a time string (dataset uses UTC time)
# sources: https://firms.modaps.eosdis.nasa.gov/content/descriptions/FIRMS_MODIS_Firehotspots.html
# https://firms.modaps.eosdis.nasa.gov/content/descriptions/FIRMS_VIIRS_Firehotspots.html
fire_data["acq_time_str"] = (
    fire_data["acq_time"]
    .astype(str)
    .str.zfill(4)
)

# Create a combined UTC datetime
fire_data["acq_datetime"] = pd.to_datetime(
    fire_data["acq_date"].dt.strftime("%Y-%m-%d")
    + " "
    + fire_data["acq_time_str"].str[:2]
    + ":"
    + fire_data["acq_time_str"].str[2:],
    utc=True
)

# Inspect
print(fire_data[[
    "acq_date",
    "acq_time",
    "acq_time_str",
    "acq_datetime"
]].head())

    acq_date  acq_time acq_time_str              acq_datetime
0 2012-01-20      1839         1839 2012-01-20 18:39:00+00:00
1 2012-01-20      1839         1839 2012-01-20 18:39:00+00:00
2 2012-01-20      1839         1839 2012-01-20 18:39:00+00:00
3 2012-01-20      1839         1839 2012-01-20 18:39:00+00:00
4 2012-01-20      1839         1839 2012-01-20 18:39:00+00:00


In [106]:
fire_data.head()

,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,type,sensor,nominal_resolution_m,acq_time_str,acq_datetime
0,33.4453,-114.6864,305.8,1.1,1.1,2012-01-20,1839,Terra,MODIS,36,6.03,292.8,4.8,D,0,MODIS,1000,1839,2012-01-20 18:39:00+00:00
1,33.4919,-114.6754,308.4,1.1,1.1,2012-01-20,1839,Terra,MODIS,59,6.03,289.1,8.2,D,0,MODIS,1000,1839,2012-01-20 18:39:00+00:00
2,32.9750,-116.5996,351.0,1.0,1.0,2012-01-20,1839,Terra,MODIS,96,6.03,288.1,66.5,D,0,MODIS,1000,1839,2012-01-20 18:39:00+00:00
3,32.9766,-116.6101,379.6,1.0,1.0,2012-01-20,1839,Terra,MODIS,100,6.03,291.0,154.9,D,0,MODIS,1000,1839,2012-01-20 18:39:00+00:00
4,33.5324,-115.9580,307.7,1.0,1.0,2012-01-20,1839,Terra,MODIS,0,6.03,292.1,3.8,D,0,MODIS,1000,1839,2012-01-20 18:39:00+00:00


In [107]:
print("Overall timestamp coverage:")
print("First:", fire_data["acq_datetime"].min())
print("Last:", fire_data["acq_datetime"].max())

print("\nTimestamp coverage by sensor:")
print(
    fire_data.groupby("sensor")["acq_datetime"]
    .agg(["min", "max", "count"])
)

Overall timestamp coverage:
First: 2012-01-20 09:43:00+00:00
Last: 2026-08-31 22:35:00+00:00

Timestamp coverage by sensor:
                             min                       max    count
sensor                                                             
MODIS  2012-01-20 18:39:00+00:00 2026-08-31 22:35:00+00:00   226541
VIIRS  2012-01-20 09:43:00+00:00 2026-07-31 21:28:00+00:00  1005803


In [108]:
# Count observations per month and sensor
monthly_counts = (
    fire_data
    .groupby([
        pd.Grouper(key="acq_datetime", freq="MS"),
        "sensor"
    ])
    .size()
    .unstack(fill_value=0)
)

print(monthly_counts.head(12))

print("\nMonthly statistics:")
print(monthly_counts.describe())

sensor                     MODIS  VIIRS
acq_datetime                           
2012-01-01 00:00:00+00:00     64    349
2012-02-01 00:00:00+00:00    253    958
2012-03-01 00:00:00+00:00    193    757
2012-04-01 00:00:00+00:00    126    783
2012-05-01 00:00:00+00:00    150   1066
2012-06-01 00:00:00+00:00    273   1459
2012-07-01 00:00:00+00:00    686   2718
2012-08-01 00:00:00+00:00   5838  26038
2012-09-01 00:00:00+00:00    829   4023
2012-10-01 00:00:00+00:00    254   1443
2012-11-01 00:00:00+00:00    349   1656
2012-12-01 00:00:00+00:00     79    598

Monthly statistics:
sensor         MODIS          VIIRS
count     176.000000     176.000000
mean     1287.164773    5714.789773
std      3190.946112   14501.812351
min        64.000000       0.000000
25%       241.250000    1266.750000
50%       384.000000    1724.500000
75%       873.250000    3457.500000
max     28089.000000  131425.000000


In [109]:
from pyproj import Transformer
import numpy as np

# Convert WGS84 latitude/longitude to California Albers (metres)
transformer = Transformer.from_crs(
    "EPSG:4326",
    "EPSG:3310",
    always_xy=True
)

fire_data["x"], fire_data["y"] = transformer.transform(
    fire_data["longitude"].values,
    fire_data["latitude"].values
)

# Assign each detection to a 1 km grid cell
fire_data["grid_x"] = np.floor(fire_data["x"] / 1000).astype("int32")
fire_data["grid_y"] = np.floor(fire_data["y"] / 1000).astype("int32")

print(fire_data[["x", "y", "grid_x", "grid_y"]].head())

               x              y  grid_x  grid_y
0  494084.345016 -493826.621385     494    -494
1  494816.139826 -488611.635473     494    -489
2  318152.299802 -554086.569536     318    -555
3  317163.969292 -553944.613678     317    -554
4  375516.204879 -490014.379853     375    -491


## Harmonization

### Aggregate the observations

In [110]:
# Aggregate the observations by day
daily_grid = (
    fire_data
    .groupby(["acq_date", "grid_x", "grid_y", "sensor"])
    .agg(
        hotspot_count=("latitude", "size"),
        mean_frp=("frp", "mean"),
        max_frp=("frp", "max"),
        mean_brightness=("brightness", "mean")
    )
    .reset_index()
)

print("Shape:", daily_grid.shape)
print(daily_grid.head())

print(daily_grid.groupby("sensor")["hotspot_count"].sum())
print("\nGrouped records per sensor:")
print(daily_grid["sensor"].value_counts())

Shape: (542444, 8)
    acq_date  grid_x  grid_y sensor  hotspot_count  mean_frp  max_frp  \
0 2012-01-20      52    -276  VIIRS              1      2.27     2.27   
1 2012-01-20     131    -214  VIIRS              1      0.90     0.90   
2 2012-01-20     132    -213  VIIRS              1      2.10     2.10   
3 2012-01-20     133    -213  VIIRS              3      1.45     1.45   
4 2012-01-20     135    -410  VIIRS              1      0.69     0.69   

   mean_brightness  
0       326.100000  
1       299.940000  
2       308.760000  
3       309.523333  
4       299.650000  
sensor
MODIS     226541
VIIRS    1005803
Name: hotspot_count, dtype: int64

Grouped records per sensor:
sensor
VIIRS    374731
MODIS    167713
Name: count, dtype: int64


In [111]:
# Find grid cells and dates observed by BOTH sensors
overlap = (
    daily_grid
    .groupby(["acq_date", "grid_x", "grid_y"])["sensor"]
    .nunique()
)

print("Cells observed by both sensors:",
      (overlap == 2).sum())

Cells observed by both sensors: 105346


In [112]:
# Keep only cells and days observed by both sensors
common_cells = overlap[overlap == 2].index

matched = (
    fire_data.set_index(["acq_date", "grid_x", "grid_y"])
    .loc[common_cells]
    .reset_index()
)

# Compare observation times for each sensor
time_comparison = (
    matched.groupby(["acq_date", "grid_x", "grid_y", "sensor"])
    ["acq_datetime"].agg(["min", "max"])
    .reset_index()
)

print(time_comparison.head(10))

    acq_date  grid_x  grid_y sensor                       min  \
0 2012-01-20     317    -554  MODIS 2012-01-20 18:39:00+00:00   
1 2012-01-20     317    -554  VIIRS 2012-01-20 09:43:00+00:00   
2 2012-01-21      11    -160  MODIS 2012-01-21 20:59:00+00:00   
3 2012-01-21      11    -160  VIIRS 2012-01-21 20:44:00+00:00   
4 2012-01-21      47    -305  MODIS 2012-01-21 20:59:00+00:00   
5 2012-01-21      47    -305  VIIRS 2012-01-21 20:44:00+00:00   
6 2012-01-23      -5    -167  MODIS 2012-01-23 19:09:00+00:00   
7 2012-01-23      -5    -167  VIIRS 2012-01-23 20:07:00+00:00   
8 2012-01-23      24    -157  MODIS 2012-01-23 20:47:00+00:00   
9 2012-01-23      24    -157  VIIRS 2012-01-23 21:47:00+00:00   

                        max  
0 2012-01-20 18:39:00+00:00  
1 2012-01-20 21:02:00+00:00  
2 2012-01-21 20:59:00+00:00  
3 2012-01-21 20:44:00+00:00  
4 2012-01-21 20:59:00+00:00  
5 2012-01-21 20:44:00+00:00  
6 2012-01-23 19:09:00+00:00  
7 2012-01-23 20:07:00+00:00  
8 2012-01-23 2

In [113]:
fire_data[
    (fire_data["acq_date"] == "2012-01-20") &
    (fire_data["grid_x"] == 317) &
    (fire_data["grid_y"] == -554)
][[
    "sensor",
    "latitude",
    "longitude",
    "acq_datetime",
    "brightness",
    "frp"
]].sort_values("acq_datetime")

,sensor,latitude,longitude,acq_datetime,brightness,frp
226560,VIIRS,32.97903,-116.60450,2012-01-20 09:43:00+00:00,339.46,6.30
3,MODIS,32.97660,-116.61010,2012-01-20 18:39:00+00:00,379.60,154.90
226562,VIIRS,32.97850,-116.60201,2012-01-20 21:02:00+00:00,337.89,6.75
226563,VIIRS,32.97794,-116.60663,2012-01-20 21:02:00+00:00,346.72,63.00
226564,VIIRS,32.97739,-116.61108,2012-01-20 21:02:00+00:00,302.24,63.00
226565,VIIRS,32.98152,-116.60719,2012-01-20 21:02:00+00:00,336.00,5.71


### Harmonize the confidence labels

In [114]:
# Harmonize the confidence labels, source: https://www.earthdata.nasa.gov/s3fs-public/imported/MODIS_C6_Fire_User_Guide_A.pdf

# Create a new column without changing the original confidence values
fire_data["confidence_class"] = None

# MODIS: convert numerical confidence into categories
modis_mask = fire_data["sensor"] == "MODIS"
modis_conf = pd.to_numeric(
    fire_data.loc[modis_mask, "confidence"],
    errors="coerce"
)

fire_data.loc[modis_mask, "confidence_class"] = np.select(
    [
        modis_conf < 30,
        (modis_conf >= 30) & (modis_conf < 80),
        (modis_conf >= 80) & (modis_conf <= 100)
    ],
    ["low", "nominal", "high"],
    default="unknown"
)

# VIIRS: map its existing categorical labels
viirs_mask = fire_data["sensor"] == "VIIRS"

fire_data.loc[viirs_mask, "confidence_class"] = (
    fire_data.loc[viirs_mask, "confidence"]
    .astype(str)
    .str.lower()
    .map({"l": "low", "n": "nominal", "h": "high"})
    .fillna("unknown")
)

In [115]:
pd.crosstab(
    fire_data["sensor"],
    fire_data["confidence_class"]
)

confidence_class,high,low,nominal
sensor,,,
MODIS,116402,14285,95854
VIIRS,72100,44044,889659


In [116]:
confidence_percentages = (
    pd.crosstab(
        fire_data["sensor"],
        fire_data["confidence_class"],
        normalize="index"
    ) * 100
).round(2)

confidence_percentages

confidence_class,high,low,nominal
sensor,,,
MODIS,51.38,6.31,42.31
VIIRS,7.17,4.38,88.45


### Find candidate MODIS–VIIRS pairs

In [117]:
# Find candidate MODIS–VIIRS pairs within 1 km and 3 hours

from scipy.spatial import cKDTree

# Separate the two sensors
modis = fire_data[fire_data["sensor"] == "MODIS"].copy()
viirs = fire_data[fire_data["sensor"] == "VIIRS"].copy()

# Build a spatial search tree for VIIRS
viirs_xy = viirs[["x", "y"]].to_numpy()
tree = cKDTree(viirs_xy)

# Convert timestamps to numeric values for fast comparison
viirs_times = viirs["acq_datetime"].astype("int64").to_numpy()
modis_times = modis["acq_datetime"].astype("int64").to_numpy()

# Matching thresholds
max_distance = 1000          # metres
max_time_difference = 3 * 60 * 60 * 1_000_000_000  # 3 hours in ns

matches = []

# Process MODIS in batches to control memory use
batch_size = 5000

for start in range(0, len(modis), batch_size):
    end = min(start + batch_size, len(modis))

    batch_xy = modis.iloc[start:end][["x", "y"]].to_numpy()

    nearby = tree.query_ball_point(
        batch_xy,
        r=max_distance,
        workers=-1
    )

    for i, viirs_indices in enumerate(nearby):
        if not viirs_indices:
            continue

        viirs_indices = np.asarray(viirs_indices)
        modis_index = start + i

        time_diffs = np.abs(
            viirs_times[viirs_indices] - modis_times[modis_index]
        )

        valid = time_diffs <= max_time_difference
        valid_indices = viirs_indices[valid]

        if len(valid_indices) == 0:
            continue

        distances = np.linalg.norm(
            viirs_xy[valid_indices] - batch_xy[i],
            axis=1
        )

        for viirs_index, distance in zip(valid_indices, distances):
            matches.append((
                modis.index[modis_index],
                viirs.index[viirs_index],
                distance,
                abs(viirs_times[viirs_index] -
                    modis_times[modis_index]) / 60_000_000_000
            ))

# Create the results table
candidate_matches = pd.DataFrame(
    matches,
    columns=[
        "modis_index",
        "viirs_index",
        "distance_m",
        "time_difference_minutes"
    ]
)

print("Candidate pairs:", len(candidate_matches))
candidate_matches.head()

Candidate pairs: 10195697


,modis_index,viirs_index,distance_m,time_difference_minutes
0,2,226559,738.953676,0.536
1,2,226560,639.985690,0.536
2,2,226561,362.716042,0.536
3,2,226562,448.555935,0.143
4,2,226563,734.086115,0.143


In [118]:
print("Total candidate pairs:", len(candidate_matches))

print("\nDistance (metres):")
print(candidate_matches["distance_m"].describe().round(2))

print("\nTime difference (minutes):")
print(candidate_matches["time_difference_minutes"].describe().round(2))

print("\nUnique MODIS detections matched:",
      candidate_matches["modis_index"].nunique())

print("Unique VIIRS detections matched:",
      candidate_matches["viirs_index"].nunique())

Total candidate pairs: 10195697

Distance (metres):
count    10195697.00
mean          654.17
std           238.47
min             0.00
25%           481.78
50%           690.83
75%           856.47
max          1000.00
Name: distance_m, dtype: float64

Time difference (minutes):
count    10195697.00
mean            5.56
std            13.55
min             0.00
25%             0.74
50%             2.11
75%             5.16
max           180.00
Name: time_difference_minutes, dtype: float64

Unique MODIS detections matched: 209138
Unique VIIRS detections matched: 846713


In [119]:
for distance in [250, 375, 500, 750, 1000]:
    count = (candidate_matches["distance_m"] <= distance).sum()
    print(f"{distance} m: {count:,} pairs ({count / len(candidate_matches) * 100:.1f}%)")

250 m: 702,292 pairs (6.9%)
375 m: 1,562,788 pairs (15.3%)
500 m: 2,739,032 pairs (26.9%)
750 m: 5,956,366 pairs (58.4%)
1000 m: 10,195,697 pairs (100.0%)


## Exporting Data

In [123]:
# Create an export copy; leave the original data untouched
fire_data_export = fire_data.copy()

# Preserve the original confidence values as text
fire_data_export["confidence"] = (
    fire_data_export["confidence"].astype(str)
)

# Also keep a numeric version where applicable (MODIS)
fire_data_export["confidence_numeric"] = pd.to_numeric(
    fire_data_export["confidence"],
    errors="coerce"
)

# Export
fire_data_export.to_parquet(
    "data/processed/fire_data.parquet", index=False
)

daily_grid.to_parquet(
    "data/processed/daily_grid.parquet", index=False
)

# Verify the exported files
check = pd.read_parquet("data/processed/fire_data.parquet")

print("Fire data:", check.shape)
print("Columns:", check.columns.tolist())
print("\nSensors:")
print(check["sensor"].value_counts())

print("\nDaily grid:", pd.read_parquet("data/processed/daily_grid.parquet").shape)

Fire data: (1232344, 25)
Columns: ['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_t31', 'frp', 'daynight', 'type', 'sensor', 'nominal_resolution_m', 'acq_time_str', 'acq_datetime', 'x', 'y', 'grid_x', 'grid_y', 'confidence_class', 'confidence_numeric']

Sensors:
sensor
VIIRS    1005803
MODIS     226541
Name: count, dtype: int64

Daily grid: (542444, 8)


In [ ]:
# from pathlib import Path
# import pandas as pd

# # Exact folder used by your Streamlit project
# output_dir = Path(
#     r"C:\Users\salma\Downloads\NASA Hackathon 26\nasa firewatch\data\processed"
# )
# output_dir.mkdir(parents=True, exist_ok=True)

# # Confirm the source data has the column
# print("Original fire_data has sensor:", "sensor" in fire_data.columns)

# # Prepare and save
# fire_data_export = fire_data.copy()
# fire_data_export["confidence"] = fire_data_export["confidence"].astype(str)
# fire_data_export["confidence_numeric"] = pd.to_numeric(
#     fire_data_export["confidence"], errors="coerce"
# )

# fire_path = output_dir / "fire_data.parquet"
# daily_path = output_dir / "daily_grid.parquet"

# fire_data_export.to_parquet(fire_path, index=False)
# daily_grid.to_parquet(daily_path, index=False)

# # Read the exact files back and verify
# check_fire = pd.read_parquet(fire_path)
# check_daily = pd.read_parquet(daily_path)

# print("\nFire file:", fire_path)
# print("Fire columns:", check_fire.columns.tolist())
# print("\nSensor counts:")
# print(check_fire["sensor"].value_counts() if "sensor" in check_fire.columns else "MISSING")
# print("\nDaily file has sensor:", "sensor" in check_daily.columns)

Original fire_data has sensor: True

Fire file: C:\Users\salma\Downloads\NASA Hackathon 26\nasa firewatch\data\processed\fire_data.parquet
Fire columns: ['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_t31', 'frp', 'daynight', 'type', 'sensor', 'nominal_resolution_m', 'acq_time_str', 'acq_datetime', 'x', 'y', 'grid_x', 'grid_y', 'confidence_class', 'confidence_numeric']

Sensor counts:
sensor
VIIRS    1005803
MODIS     226541
Name: count, dtype: int64

Daily file has sensor: True
